In [0]:
# ======================================================================
# CMD 04.01 — LOAD H₂ THERMODYNAMIC MODELING DATA
# ======================================================================

import numpy as np
import pandas as pd

hydrogen_train = spark.table(
    "ml_perovskites.ml_hydrogen_thermo_train"
).toPandas()

hydrogen_test = spark.table(
    "ml_perovskites.ml_hydrogen_thermo_test"
).toPandas()

print("Train shape:", hydrogen_train.shape)
print("Test shape:", hydrogen_test.shape)

In [0]:
# ======================================================================
# CMD 04.02 — DEFINE H₂ FEATURE GROUPS
# ======================================================================

material_features = [
    col for col in hydrogen_train.columns
    if col.startswith("mean_")
]

thermo_features = [
    "thermo_mean_MW",
    "thermo_mean_Tm",
    "thermo_mean_Tb",
    "thermo_mean_Hfus",
    "thermo_mean_Hvap"
]

target_column = "hydrogen_target"

print("Material features:", len(material_features))
print("Thermodynamic features:", len(thermo_features))
print("Total candidate features:", len(material_features) + len(thermo_features))

In [0]:
# ======================================================================
# CMD 04.03 — AUDIT FEATURE VARIANCE
# ======================================================================

X_train_candidates = hydrogen_train[
    material_features + thermo_features
].copy()

feature_variance = X_train_candidates.var(
    axis=0,
    ddof=0
).sort_values()

variance_audit = pd.DataFrame({
    "feature": feature_variance.index,
    "variance": feature_variance.values
})

print("Candidate features:", len(variance_audit))
print("Zero-variance features:", (variance_audit["variance"] == 0).sum())

display(variance_audit)

In [0]:
# ======================================================================
# CMD 04.04 — AUDIT FEATURE CORRELATION
# ======================================================================

correlation_matrix = X_train_candidates.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "absolute_correlation"
]

correlation_pairs = correlation_pairs.sort_values(
    "absolute_correlation",
    ascending=False
).reset_index(drop=True)

high_correlation_pairs = correlation_pairs[
    correlation_pairs["absolute_correlation"] >= 0.90
]

print(
    "Pairs with |r| >= 0.90:",
    len(high_correlation_pairs)
)

display(high_correlation_pairs)

In [0]:
# ======================================================================
# CMD 04.05 — DEFINE CORRELATION-REDUNDANT FEATURES
# ======================================================================

CORRELATION_THRESHOLD = 0.90

correlation_redundant_features = set()

for _, row in high_correlation_pairs.iterrows():
    correlation_redundant_features.add(row["feature_2"])

correlation_redundant_features = sorted(
    correlation_redundant_features
)

print(
    "Correlation-redundant features:",
    len(correlation_redundant_features)
)

display(
    high_correlation_pairs[
        [
            "feature_1",
            "feature_2",
            "absolute_correlation"
        ]
    ]
)

In [0]:
# ======================================================================
# CMD 04.06 — REMOVE CORRELATION-REDUNDANT FEATURES
# ======================================================================

selected_features_corr = [
    feature
    for feature in X_train_candidates.columns
    if feature not in correlation_redundant_features
]

X_train_corr = X_train_candidates[selected_features_corr].copy()
X_test_corr = hydrogen_test[selected_features_corr].copy()

print("Original features:", X_train_candidates.shape[1])
print("Removed features:", len(correlation_redundant_features))
print("Remaining features:", len(selected_features_corr))

print("\nRemoved:")
for feature in correlation_redundant_features:
    print("-", feature)

In [0]:
# ======================================================================
# CMD 04.07 — RANK FEATURES BY MUTUAL INFORMATION
# ======================================================================

from sklearn.feature_selection import mutual_info_regression

# Use the H₂ target directly from the training dataset
y_train_mi = np.log1p(
    hydrogen_train[target_column]
)

mi_scores = mutual_info_regression(
    X_train_corr,
    y_train_mi,
    random_state=42
)

mi_ranking = (
    pd.DataFrame({
        "feature": X_train_corr.columns,
        "mutual_information": mi_scores
    })
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(mi_ranking)

In [0]:
# ======================================================================
# CMD 04.08 — DEFINE MUTUAL INFORMATION FEATURE SUBSETS
# ======================================================================

MI_TOP_K = [10, 15, 20, 25, 30, 35, 41]

mi_feature_sets = {
    k: mi_ranking.head(k)["feature"].tolist()
    for k in MI_TOP_K
}

for k, features in mi_feature_sets.items():
    print(f"Top {k}: {len(features)} features")

In [0]:
# ======================================================================
# CMD 04.09 — CROSS-VALIDATE MI FEATURE SUBSETS ON LOG TARGET
# ======================================================================

from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Recreate target explicitly
y_train_mi = np.log1p(
    hydrogen_train[target_column]
)

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

mi_cv_results = []

for k, features in mi_feature_sets.items():

    X_subset = X_train_corr[features]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("krr", KernelRidge(
            alpha=1.0,
            kernel="rbf"
        ))
    ])

    scores = cross_val_score(
        model,
        X_subset,
        y_train_mi,
        cv=cv,
        scoring="r2"
    )

    mi_cv_results.append({
        "n_features": k,
        "mean_cv_r2": scores.mean(),
        "std_cv_r2": scores.std()
    })

mi_cv_results = pd.DataFrame(mi_cv_results)

display(
    mi_cv_results.sort_values(
        "mean_cv_r2",
        ascending=False
    )
)

In [0]:
# ======================================================================
# CMD 04.10 — COMPARE FEATURE SUBSETS ACROSS MODELS
# ======================================================================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Recreate log-transformed H₂ target explicitly
# ----------------------------------------------------------------------

y_train_mi = np.log1p(
    hydrogen_train[target_column]
)

# ----------------------------------------------------------------------
# Cross-validation
# ----------------------------------------------------------------------

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ----------------------------------------------------------------------
# Models
# ----------------------------------------------------------------------

models = {
    "KRR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KernelRidge(
            alpha=1.0,
            kernel="rbf"
        ))
    ]),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(
            kernel="rbf",
            C=10.0,
            epsilon=0.1
        ))
    ]),

    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    )
}

# ----------------------------------------------------------------------
# Evaluate
# ----------------------------------------------------------------------

multi_model_results = []

for k, features in mi_feature_sets.items():

    X_subset = X_train_corr[features]

    for model_name, model in models.items():

        scores = cross_val_score(
            model,
            X_subset,
            y_train_mi,
            cv=cv,
            scoring="r2"
        )

        multi_model_results.append({
            "model": model_name,
            "n_features": k,
            "mean_cv_r2": scores.mean(),
            "std_cv_r2": scores.std()
        })

multi_model_results = pd.DataFrame(
    multi_model_results
)

display(
    multi_model_results.sort_values(
        "mean_cv_r2",
        ascending=False
    )
)

In [0]:
# ======================================================================
# CMD 04.11 — OPTIMIZE RANDOM FOREST WITH MI-41 FEATURES
# ======================================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# ----------------------------------------------------------------------
# Recreate log-transformed H₂ target explicitly
# ----------------------------------------------------------------------

y_train_mi = np.log1p(
    hydrogen_train[target_column]
)

# ----------------------------------------------------------------------
# MI-41 feature set
# ----------------------------------------------------------------------

mi_41_features = mi_feature_sets[41]

X_train_mi41 = X_train_corr[mi_41_features]

# ----------------------------------------------------------------------
# Random Forest
# ----------------------------------------------------------------------

rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [300, 500, 800],
    "max_depth": [None, 5, 10, 15],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.7, 1.0]
}

# ----------------------------------------------------------------------
# Randomized search using training data only
# ----------------------------------------------------------------------

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=cv,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    refit=True
)

rf_search.fit(
    X_train_mi41,
    y_train_mi
)

print("Best CV R²:", rf_search.best_score_)
print("\nBest parameters:")
print(rf_search.best_params_)

In [0]:
# ======================================================================
# CMD 04.12 — FINAL RF-41 TEST EVALUATION
# ======================================================================

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# ----------------------------------------------------------------------
# Recreate target explicitly
# ----------------------------------------------------------------------

y_train_final = np.log1p(
    hydrogen_train[target_column]
)

y_test_original = hydrogen_test[target_column].values

# ----------------------------------------------------------------------
# Final feature matrices
# ----------------------------------------------------------------------

mi_41_features = mi_feature_sets[41]

X_train_final = X_train_corr[mi_41_features]
X_test_final = X_test_corr[mi_41_features]

# ----------------------------------------------------------------------
# Final optimized Random Forest
# ----------------------------------------------------------------------

final_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

final_rf.fit(
    X_train_final,
    y_train_final
)

# ----------------------------------------------------------------------
# Predictions
# ----------------------------------------------------------------------

y_pred_log = final_rf.predict(X_test_final)

# Convert back to original H₂ scale
y_pred_original = np.expm1(y_pred_log)

# ----------------------------------------------------------------------
# Metrics — original H₂ scale
# ----------------------------------------------------------------------

test_r2 = r2_score(
    y_test_original,
    y_pred_original
)

test_mae = mean_absolute_error(
    y_test_original,
    y_pred_original
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test_original,
        y_pred_original
    )
)

print("Final RF-41 — original H₂ scale")
print("--------------------------------")
print(f"MAE : {test_mae}")
print(f"RMSE: {test_rmse}")
print(f"R²  : {test_r2}")

In [0]:
# ======================================================================
# CMD 04.13 — TEST PREDICTION AUDIT
# ======================================================================

prediction_audit = pd.DataFrame({
    "actual_h2": y_test_original,
    "predicted_h2": y_pred_original
})

prediction_audit["absolute_error"] = (
    prediction_audit["actual_h2"] -
    prediction_audit["predicted_h2"]
).abs()

prediction_audit["log_actual"] = np.log1p(
    prediction_audit["actual_h2"]
)

prediction_audit["log_predicted"] = np.log1p(
    prediction_audit["predicted_h2"]
)

print("Largest absolute errors:")
display(
    prediction_audit
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .head(15)
)

print("\nTest target statistics:")
display(
    prediction_audit["actual_h2"].describe()
)

print("\nPrediction statistics:")
display(
    prediction_audit["predicted_h2"].describe()
)

In [0]:
# ======================================================================
# CMD 04.14 — EVALUATE RFE-10 FEATURE SUBSET
# ======================================================================

rfe_features = rfe_ranking.loc[
    rfe_ranking["selected"],
    "feature"
].tolist()

X_train_rfe = X_train_corr[rfe_features]

rfe_scores = cross_val_score(
    models["KRR"],
    X_train_rfe,
    y_train_h2_log,
    cv=cv,
    scoring="r2"
)

print("RFE features:", len(rfe_features))
print("Mean CV R²:", rfe_scores.mean())
print("Std CV R²:", rfe_scores.std())

print("\nSelected features:")
for feature in rfe_features:
    print("-", feature)

In [0]:
# ======================================================================
# CMD 04.15 — EVALUATE RFE-10 WITH RANDOM FOREST
# ======================================================================

rf_rfe_cv = cross_val_score(
    rf_rfe_estimator,
    X_train_rfe,
    y_train_h2_log,
    cv=cv,
    scoring="r2"
)

print("RFE-10 Random Forest")
print("Mean CV R²:", rf_rfe_cv.mean())
print("Std CV R²:", rf_rfe_cv.std())

In [0]:
# ======================================================================
# CMD 04.16 — FINAL TRAINING-CV FEATURE COMPARISON
# ======================================================================

mi_41_features = mi_ranking.head(41)["feature"].tolist()

comparison_features = {
    "MI_41": mi_41_features,
    "RFE_10": rfe_features
}

comparison_results = []

for name, features in comparison_features.items():

    scores = cross_val_score(
        rf_rfe_estimator,
        X_train_corr[features],
        y_train_h2_log,
        cv=cv,
        scoring="r2"
    )

    comparison_results.append({
        "feature_set": name,
        "n_features": len(features),
        "mean_cv_r2": scores.mean(),
        "std_cv_r2": scores.std()
    })

comparison_results = pd.DataFrame(comparison_results)

display(comparison_results)

In [0]:
# ======================================================================
# CMD 04.17 — EVALUATE MI-41 ON UNTOUCHED TEST SET
# ======================================================================

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

mi_41_features = mi_ranking.head(41)["feature"].tolist()

X_train_mi41 = X_train_corr[mi_41_features]
X_test_mi41 = hydrogen_test[mi_41_features]

final_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

final_rf.fit(
    X_train_mi41,
    y_train_h2_log
)

y_pred_log = final_rf.predict(X_test_mi41)
y_pred = np.expm1(y_pred_log)

test_r2 = r2_score(
    hydrogen_test[target_column],
    y_pred
)

test_mae = mean_absolute_error(
    hydrogen_test[target_column],
    y_pred
)

test_rmse = mean_squared_error(
    hydrogen_test[target_column],
    y_pred
) ** 0.5

print("Test R²:", test_r2)
print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

In [0]:
# ======================================================================
# CMD 04.18 — AUDIT TEST PREDICTIONS AND TARGET DISTRIBUTION
# ======================================================================

prediction_audit = pd.DataFrame({
    "actual_h2": hydrogen_test[target_column].values,
    "predicted_h2": y_pred
})

print("Test target statistics:")
display(
    hydrogen_test[target_column].describe()
)

print("\nPrediction statistics:")
display(
    prediction_audit["predicted_h2"].describe()
)

print("\nLargest absolute errors:")
prediction_audit["absolute_error"] = (
    prediction_audit["actual_h2"] -
    prediction_audit["predicted_h2"]
).abs()

display(
    prediction_audit
    .sort_values("absolute_error", ascending=False)
    .head(15)
)

In [0]:
# ======================================================================
# CMD 04.19 — COMPARE TRAIN/TEST H₂ DISTRIBUTION
# ======================================================================

distribution_comparison = pd.DataFrame({
    "statistic": [
        "count",
        "zero_count",
        "minimum",
        "25th_percentile",
        "median",
        "75th_percentile",
        "90th_percentile",
        "95th_percentile",
        "99th_percentile",
        "maximum"
    ],
    "train": [
        len(hydrogen_train),
        (hydrogen_train[target_column] == 0).sum(),
        hydrogen_train[target_column].min(),
        hydrogen_train[target_column].quantile(0.25),
        hydrogen_train[target_column].median(),
        hydrogen_train[target_column].quantile(0.75),
        hydrogen_train[target_column].quantile(0.90),
        hydrogen_train[target_column].quantile(0.95),
        hydrogen_train[target_column].quantile(0.99),
        hydrogen_train[target_column].max()
    ],
    "test": [
        len(hydrogen_test),
        (hydrogen_test[target_column] == 0).sum(),
        hydrogen_test[target_column].min(),
        hydrogen_test[target_column].quantile(0.25),
        hydrogen_test[target_column].median(),
        hydrogen_test[target_column].quantile(0.75),
        hydrogen_test[target_column].quantile(0.90),
        hydrogen_test[target_column].quantile(0.95),
        hydrogen_test[target_column].quantile(0.99),
        hydrogen_test[target_column].max()
    ]
})

display(distribution_comparison)